In [1]:
import os
import pandas as pd
from maomao.parsing.parsing_utils import *
from maomao.utils.constants import *

#### Processing and standardizing peptide datasets (QSVM-PEPTIDE)

This notebook curates the **QSVM-PEPTIDE** dataset by integrating multiple hemolytic peptide collections derived from different feature representations, performing duplicate consistency checks, generating metadata, and exporting a standardized dataset for downstream machine learning tasks.

 - **Toxic effect / endpoint:** hemolytic
- **Source:** QSVM-PEPTIDE
- **Sequence scope:** only non-modified peptide sequences are retained for the final dataset.

The pipeline performs the following steps:

- **Parses QSVM-PEPTIDE data sources** corresponding to two feature dimensionalities:
  - **Hemo40D** datasets (three independent splits with train/test partitions),
  - **Hemo56D** datasets (three independent data groups).
- **Standardizes sequence and label columns** across heterogeneous CSV formats.
- **Infers labels when necessary**:
  - files containing `"neg"` in their name are assigned `label = 0`,
  - all other files are assigned `label = 1`.
- **Concatenates all datasets** into a single unified hemolytic peptide collection.
- **Performs duplicate sequence checks**:
  - consistent duplicates are merged,
  - conflicting label assignments are detected and exported as errors.
- **Builds dataset metadata** using the project-wide raw data description file.
- **Exports curated outputs**:
  - `processed_hemolytic_dataset.csv`
  - `detected_error_sequences.csv`
  - `metadata.json`

In [2]:
name_source = "QSVM-PEPTIDE"
name_task = "toxic_effect_classification"

# PATH_INPUT and PATH_EXPORT are imported from maomao.utils.constants
# Update them in constants.py according to the required input and export paths.

- Reading raw data

In [3]:
df_QSVM_Hemo40D = pd.concat([
    pd.read_csv(os.path.join(folder, file))
    .assign(label=0 if 'neg' in file else 1)
    .rename(columns={"seq": "sequence"})[["sequence", "label"]]
    for folder in [f"{PATH_INPUT}/{name_source}/Hemo40D/data1/test/", 
                   f"{PATH_INPUT}/{name_source}/Hemo40D/data1/train/",
                   f"{PATH_INPUT}/{name_source}/Hemo40D/data2/test/", 
                   f"{PATH_INPUT}/{name_source}/Hemo40D/data2/train/",
                   f"{PATH_INPUT}/{name_source}/Hemo40D/data3/test/", 
                   f"{PATH_INPUT}/{name_source}/Hemo40D/data3/train/"] 
    for file in os.listdir(folder)])

In [4]:
df_QSVM_Hemo56D = pd.concat([
    pd.read_csv(os.path.join(folder, file))
    .rename(columns={"Sequence": "sequence", "Label": "label"})[["sequence", "label"]]
    for folder in [f"{PATH_INPUT}/{name_source}/Hemo56D/data1/", 
                   f"{PATH_INPUT}/{name_source}/Hemo56D/data1/",
                   f"{PATH_INPUT}/{name_source}/Hemo56D/data2/", 
                   f"{PATH_INPUT}/{name_source}/Hemo56D/data2/",
                   f"{PATH_INPUT}/{name_source}/Hemo56D/data3/", 
                   f"{PATH_INPUT}/{name_source}/Hemo56D/data3/"] 
    for file in os.listdir(folder)])

- Concatenate dataset

In [5]:
df_QSVM = pd.concat([
    df_QSVM_Hemo40D, 
    df_QSVM_Hemo56D
],ignore_index=True)

df_QSVM.shape

(11221, 2)

- Checking duplicates

In [6]:
df_remove_duplicated, df_errors, df_unique = processing_duplicated(df_QSVM, group_seq="sequence", sort_key="label")
df_full = pd.concat([df_unique, df_remove_duplicated], axis=0)

In [7]:
df_full.shape

(2195, 2)

In [8]:
df_errors.shape

(85, 1)

- Working with metada

In [9]:
df_metada = read_metadata("../../raw_data/raw_data_description.xlsx", name_source)
dict_metadata = create_metada_with_multiple_values(df_metada)

- Exporting data

In [10]:
dict_metadata.update({
    "number_of_raw_sequences": int(len(df_QSVM)),
    "number_of_sequences_retained": len(df_full),
    "number_of_positive_sequences": int((df_full["label"] == 1).sum()),
    "number_of_negative_sequences": int((df_full["label"] == 0).sum()),
    "number_of_erroneous_sequences": int(len(df_errors)),
    "modified_sequences_included": False,
})

dict_metadata

{'type source': 'Dataset',
 'static-dynamic': 'Static',
 'license': 'No information',
 'year of publication': 2024,
 'last update date': datetime.datetime(2024, 9, 19, 0, 0),
 'download date': Timestamp('2025-10-17 00:00:00'),
 'file format': 'csv',
 'peptide property': 'hemolytic, toxic',
 'dataset information': 'Negative;Positive;Positive, Negative',
 'unit of measurement': 'No information',
 'obtaining negative dataset': 'Sampling from Swiss-Prot, Sampling from another DB',
 'repository or server': 'https://github.com/sxxnz/QSVM-PEPTIDE/tree/main/data',
 'publication': 'https://link.springer.com/article/10.1007/s11128-024-04540-5',
 'number_of_raw_sequences': 11221,
 'number_of_sequences_retained': 2195,
 'number_of_positive_sequences': 911,
 'number_of_negative_sequences': 1284,
 'number_of_erroneous_sequences': 85,
 'modified_sequences_included': False}

In [11]:
os.makedirs(f"{PATH_EXPORT}/{name_task}/{name_source}/", exist_ok=True)
export_json(f"{PATH_EXPORT}/{name_task}/{name_source}/metadata.json", dict_metadata)

In [12]:
df_full.to_csv(f"{PATH_EXPORT}/{name_task}/{name_source}/processed_hemolytic_dataset.csv", index=False)
df_errors.to_csv(f"{PATH_EXPORT}/{name_task}/{name_source}/detected_error_sequences.csv", index=False)